
# Dataflow Flex Templates

A team has a small, genuinely useful Beam pipeline — reads a text file, counts word frequency.
Right now running it means checking out the source, matching the Python environment, and running
`python pipeline.py` by hand. Package it once as a **Flex Template** — a container image plus a
small JSON spec — and afterward anyone with the right IAM permissions can launch it with just a
job name and two parameters. No source code, no environment, on their end.

## Time budget
| Step | What happens | ~Time |
|---|---|---|
| Build | Cloud Build containerizes the pipeline | 3-6 min |
| Run | A real Dataflow job spins up, runs, shuts down | 5-10 min |

Both waits are the actual mechanism being demonstrated, not overhead — same pattern as every
other provisioning-heavy step elsewhere in this course.


## 1. Auth

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("authenticated")
else:
    print("not in Colab — assuming ADC is set up")


## 2. Config, bucket, Artifact Registry repo

In [ ]:
import time

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
REGION = input("Region [us-central1]: ").strip() or "us-central1"

_suffix = int(time.time())
BUCKET_NAME = f"{PROJECT_ID}-flex-template-lab-{_suffix}"
BUCKET_URI = f"gs://{BUCKET_NAME}"
REPO_NAME = "dataflow-templates"
IMAGE_NAME = f"wordcount-{_suffix}"

!gcloud config set project {PROJECT_ID}
!gcloud services enable dataflow.googleapis.com cloudbuild.googleapis.com \
    artifactregistry.googleapis.com compute.googleapis.com --project={PROJECT_ID}

!gsutil mb -l {REGION} {BUCKET_URI}

!gcloud artifacts repositories create {REPO_NAME} \
    --repository-format=docker --location={REGION} --project={PROJECT_ID} \
    --description="Dataflow Flex Template images" --quiet

print("bucket:", BUCKET_URI, "| repo:", REPO_NAME)



### Grant the worker service account what it actually needs
Dataflow workers run as the **Compute Engine default service account** unless you specify one
explicitly. On newer projects, that account no longer auto-gets the Editor role it used to (a
2024+ policy change — the same thing that broke a Cloud Function deploy earlier in this course).
Without `roles/dataflow.worker` and write access to our bucket, the job **launches fine but fails
once workers actually try to start** — exactly the "created, then fails" pattern this section
exists to prevent.


In [ ]:
project_number = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
project_number = project_number[0].strip()
compute_sa = f"{project_number}-compute@developer.gserviceaccount.com"

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{compute_sa}" --role="roles/dataflow.worker" --quiet

!gsutil iam ch serviceAccount:{compute_sa}:roles/storage.objectAdmin {BUCKET_URI}

print("granted dataflow.worker + storage access to", compute_sa)
print("waiting 30s for IAM to propagate...")
import time as _t
_t.sleep(30)



> 🖥️ IAM & Admin > IAM → the compute default service account should show `Dataflow Worker`. Cloud
> Storage > your bucket > Permissions → same account should show `Storage Object Admin`.


## 3. The pipeline, requirements, metadata

In [ ]:
sample_text = '''Apache Beam is a unified model for batch and streaming data processing.
Dataflow runs Apache Beam pipelines on managed infrastructure.
A Flex Template packages a Beam pipeline as a container image.
Anyone can launch a Flex Template without needing the pipeline source code.
Apache Beam Apache Beam Dataflow Dataflow Dataflow
'''

with open("input.txt", "w") as f:
    f.write(sample_text)

!gsutil cp input.txt {BUCKET_URI}/input.txt
print("uploaded input.txt")



Two things make this Flex-Template-ready rather than just an ordinary Beam script:
`add_value_provider_argument` (so `input_path`/`output_path` are set at *launch* time, not baked
in now), and calling `pipeline.run()` directly instead of `with beam.Pipeline(...) as p:` — that
context manager blocks waiting for completion, which is fatal for a launcher that needs to exit
right after submitting the job.


In [ ]:
import os
os.makedirs("flex_template", exist_ok=True)

with open("flex_template/pipeline.py", "w") as f:
    f.write('''
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions


class TemplateOptions(PipelineOptions):
    @classmethod
    def _add_argparse_args(cls, parser):
        parser.add_value_provider_argument("--input_path", type=str)
        parser.add_value_provider_argument("--output_path", type=str)


def run():
    pipeline_options = PipelineOptions()
    template_options = pipeline_options.view_as(TemplateOptions)

    pipeline = beam.Pipeline(options=pipeline_options)
    (
        pipeline
        | "ReadLines" >> beam.io.ReadFromText(template_options.input_path)
        | "ExtractWords" >> beam.FlatMap(lambda line: line.split())
        | "CountWords" >> beam.combiners.Count.PerElement()
        | "FormatResult" >> beam.Map(lambda kv: f"{kv[0]}: {kv[1]}")
        | "WriteResult" >> beam.io.WriteToText(template_options.output_path)
    )
    # must call run() and exit -- NOT `with beam.Pipeline(...) as p:`, which blocks
    pipeline.run()


if __name__ == "__main__":
    run()
''')

with open("flex_template/requirements.txt", "w") as f:
    f.write("apache-beam[gcp]==2.60.0\n")

print("wrote pipeline.py + requirements.txt")


In [ ]:
import json

metadata = {
    "name": "Word Count Flex Template",
    "description": "Reads a text file, counts word frequency, writes the result to GCS.",
    "parameters": [
        {"name": "input_path", "label": "Input file", "helpText": "GCS path to input text", "isOptional": False},
        {"name": "output_path", "label": "Output prefix", "helpText": "GCS path prefix for output", "isOptional": False},
    ],
}

with open("flex_template/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("wrote metadata.json")



## 4. Build the template
No hand-written Dockerfile — `--py-path` + `--flex-template-base-image PYTHON3` containerizes
automatically. Slowest step so far: 3-6 minutes.


In [ ]:
TEMPLATE_SPEC_PATH = f"{BUCKET_URI}/templates/wordcount.json"
IMAGE_PATH = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{IMAGE_NAME}:latest"

!gcloud dataflow flex-template build {TEMPLATE_SPEC_PATH} \
    --image-gcr-path {IMAGE_PATH} \
    --sdk-language PYTHON \
    --flex-template-base-image PYTHON3 \
    --metadata-file flex_template/metadata.json \
    --py-path flex_template \
    --env FLEX_TEMPLATE_PYTHON_PY_FILE=pipeline.py \
    --env FLEX_TEMPLATE_PYTHON_REQUIREMENTS_FILE=requirements.txt \
    --project={PROJECT_ID}

print("\ntemplate spec:", TEMPLATE_SPEC_PATH)
print("image:", IMAGE_PATH)



> 🖥️ Cloud Build > History shows the build. Artifact Registry > dataflow-templates shows the
> pushed image. Cloud Storage > your bucket > templates/ shows wordcount.json.


## Cleanup

In [ ]:
CONFIRM_DELETE = True  # <-- flip to True when ready

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing happens.")


In [ ]:
if CONFIRM_DELETE:
    try:
        !gcloud dataflow jobs cancel {JOB_NAME} --region={REGION} --project={PROJECT_ID} --quiet
        print("cancelled (no-op if already done)")
    except Exception as e:
        print("job cancel skipped/failed:", e)

    try:
        !gcloud artifacts repositories delete {REPO_NAME} --location={REGION} --project={PROJECT_ID} --quiet
        print("deleted repo:", REPO_NAME)
    except Exception as e:
        print("repo cleanup skipped/failed:", e)

    try:
        !gsutil -m rm -r {BUCKET_URI}
        print("deleted bucket:", BUCKET_URI)
    except Exception as e:
        print("bucket cleanup skipped/failed:", e)

    print("\ncleanup complete")



> 🖥️ Cloud Storage, Artifact Registry, and Dataflow > Jobs should show nothing left running or
> billing (the job itself stays listed as history — jobs are never deleted, only cancelled/done).
